# MediaPipe 3D Pose — Reference Generation Notebook

## Overview
This notebook uses **MediaPipe Pose**, which outputs
`(x, y, z)` per keypoint in **normalized world coordinates**.

### Why MediaPipe instead of YOLO here?
- YOLO gives 2D `(x, y)` only → all measurements are perspective-dependent
- MediaPipe gives `(x, y, z)` in metric-scale world space → angles and distances are
  camera-angle robust
- The `z` axis = **depth** → we can now detect forward lean, elbow behind torso,
  and knee collapse in true 3D space

### What is preserved from the YOLO notebook?
- **Exact same metrics** (elbow ratio, trunk lean, hip alignment, knee width,
  back angle, elbow angle, elbow offset)
- **Exact same threshold strategy** (percentile-based)
- **Exact same dataset split**: videos for curl/pushdown/press/plank/pushup,
  frames dataset for squat/deadlift variants
- **Phase detection** (TOP / MID / BOTTOM) for squat/deadlift
- **Same output JSON structure** so corrector files need minimal changes

### What is new / extended?
- All distances now use `sqrt(dx²+dy²+dz²)` (3D Euclidean)
- All angles now use 3D dot product on `(x,y,z)` vectors
- **Trunk lean** now uses Z-axis depth: `(shoulder_z - hip_z) / torso_3d`
  instead of X-axis pixel offset → much more reliable from any camera angle
- **Hip alignment** (plank) uses 3D point-to-line distance
- **Elbow depth offset** added for press: `(elbow_z - shoulder_z) / torso_3d`
- **Knee collapse** now uses 3D knee/ankle width ratio
- Graceful fallback: if `z` is all-zero, system falls back to 2D mode

### MediaPipe ↔ YOLO keypoint index mapping
MediaPipe uses different landmark indices than YOLO COCO format.
We define a mapping so the rest of the pipeline stays consistent.

```
YOLO idx → MediaPipe landmark
  0  nose          → 0  NOSE
  5  left_shoulder → 11 LEFT_SHOULDER
  6  right_shoulder→ 12 RIGHT_SHOULDER
  7  left_elbow    → 13 LEFT_ELBOW
  8  right_elbow   → 14 RIGHT_ELBOW
  9  left_wrist    → 15 LEFT_WRIST
 10  right_wrist   → 16 RIGHT_WRIST
 11  left_hip      → 23 LEFT_HIP
 12  right_hip     → 24 RIGHT_HIP
 13  left_knee     → 25 LEFT_KNEE
 14  right_knee    → 26 RIGHT_KNEE
 15  left_ankle    → 27 LEFT_ANKLE
 16  right_ankle   → 28 RIGHT_ANKLE
```


In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────
!pip install mediapipe==0.10.14 opencv-python-headless numpy pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 51.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 25.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.25 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.3

In [2]:
import os
import json
import cv2
import numpy as np
from collections import defaultdict
import mediapipe as mp

mp_pose = mp.solutions.pose

pose_static = mp_pose.Pose(
    static_image_mode=True,
    model_complexity=2,
    enable_segmentation=False,
    smooth_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

pose_video = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,
    enable_segmentation=False,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# MediaPipe landmark indices (replaces YOLO COCO indices)
IDX = {
    "nose":           0,
    "left_shoulder":  11, "right_shoulder": 12,
    "left_elbow":     13, "right_elbow":    14,
    "left_wrist":     15, "right_wrist":    16,
    "left_hip":       23, "right_hip":      24,
    "left_knee":      25, "right_knee":     26,
    "left_ankle":     27, "right_ankle":    28,
}

CONF_THRESHOLD = 0.5

2026-04-04 10:05:19.394370: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775297119.576139      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775297119.629398      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775297120.062514      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775297120.062574      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775297120.062577      55 computation_placer.cc:177] computation placer alr

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [3]:
# ── Cell 3: Core 3D math utilities ────────────────────────────
"""
All math functions support both 2D (x,y) and 3D (x,y,z) arrays.
Fallback to 2D if z is unavailable or zero.
"""

def dist3d(a, b):
    return float(np.linalg.norm(np.array(a) - np.array(b))) + 1e-6

def angle3d(a, b, c):
    a, b, c = np.array(a, dtype=float), np.array(b, dtype=float), np.array(c, dtype=float)
    ba = a - b
    bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))

def point_to_line_dist2d(point, line_a, line_b):
    """
    2D hip alignment deviation.
    point  = hip
    line_a = shoulder
    line_b = ankle
    positive = hip below line = sagging
    negative = hip above line = piking
    Normalized by 3D body length (dist3d still used for better scale).
    """
    body_length = dist3d(line_a, line_b)
    t = (point[0] - line_a[0]) / (line_a[0] - line_b[0] + 1e-6)  # wait, wrong sign
    t = (point[0] - line_a[0]) / (line_b[0] - line_a[0] + 1e-6)
    if 0.0 <= t <= 1.0:
        expected_y = line_a[1] + t * (line_b[1] - line_a[1])
        return round(float((point[1] - expected_y) / body_length), 4)
    return None

def has_depth(xyz):
    """Returns True if z-values are non-trivial (3D data available)."""
    return xyz.shape[-1] == 3 and not np.allclose(xyz[:, 2], 0.0, atol=1e-4)


print("3D math utilities loaded.")

3D math utilities loaded.


In [4]:
def extract_mp_keypoints(image_bgr, pose_instance):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    results   = pose_instance.process(image_rgb)

    if not results.pose_world_landmarks:
        return None, None

    world_lm = results.pose_world_landmarks.landmark
    n   = len(world_lm)
    xyz = np.zeros((n, 3), dtype=float)
    vis = np.zeros(n,      dtype=float)

    for i, lm in enumerate(world_lm):
        xyz[i] = [lm.x, lm.y, lm.z]
        vis[i] = lm.visibility

    return xyz, vis


def kp(xyz, vis, idx, threshold=CONF_THRESHOLD):
    if vis[idx] >= threshold:
        return xyz[idx]
    return None


def pick_side(vis, left_idx, right_idx):
    return bool(vis[left_idx] >= vis[right_idx])

In [5]:
# ── Cell 5: Video helper ──────────────────────────────────────
VIDEO_DATASET_PATH="/kaggle/input/datasets/hasyimabdillah/workoutfitness-video"

def get_videos(folder, n_train=5, n_test=1, base_path=VIDEO_DATASET_PATH):
    p = os.path.join(base_path, folder)
    videos = sorted([
        os.path.join(p, f)
        for f in os.listdir(p)
        if f.endswith(".mp4")
    ])
    return videos[:n_train], videos[n_train:n_train + n_test]


def iter_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  Cannot open: {video_path}")
        return
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        yield frame
    cap.release()


def iter_images(exercise_path):
    """Yields image paths from frames dataset (squat/deadlift)."""
    train_path = os.path.join(exercise_path, "images", "train")
    if os.path.exists(train_path):
        for fn in os.listdir(train_path):
            if os.path.splitext(fn)[1].lower() :
                yield os.path.join(train_path, fn)
    else:
        for dirpath, _, filenames in os.walk(exercise_path):
            if "labels" in dirpath:
                continue
            for fn in filenames:
                if os.path.splitext(fn)[1].lower() :
                    yield os.path.join(dirpath, fn)


print("Video/image helpers loaded.")


Video/image helpers loaded.


---
## Part 1: Curl / Pushdown — Video Dataset
*(hammer curl, barbell biceps curl, tricep pushdown)*

### Metrics
1. **Elbow body ratio** (3D): `dist3d(elbow, hip) / dist3d(shoulder, hip)` — elbow drift
2. **Trunk lean** (3D depth-based): `(shoulder_z - hip_z) / torso_3d` — forward/back lean
3. **Elbow lateral drift** (NEW 3D): `|elbow_x - hip_x| / torso_3d` — side drift


In [19]:
# ── Cell 6: Curl/Pushdown — elbow body ratio (3D) ─────────────

def compute_elbow_body_ratio_3d(video_path):
    """
    3D UPGRADE of compute_elbow_body_ratio.
    ratio = dist3d(elbow, hip) / dist3d(shoulder, hip)
    
    NEW vs YOLO: uses world-coordinate 3D distances.
    Eliminates perspective distortion from camera angle.
    Also computes elbow_z_offset for depth-drift detection.
    """
    ratios    = []
    z_offsets = []  # NEW: elbow depth offset relative to hip

    for frame in iter_video_frames(video_path):
        xyz, vis,  = extract_mp_keypoints(frame, pose_video)
        if xyz is None:
            ratios.append(None)
            z_offsets.append(None)
            continue

        # Pick side with higher visibility
        use_left = pick_side(vis, IDX["left_elbow"], IDX["right_elbow"])
        sh_idx  = IDX["left_shoulder"]  if use_left else IDX["right_shoulder"]
        el_idx  = IDX["left_elbow"]     if use_left else IDX["right_elbow"]
        hi_idx  = IDX["left_hip"]       if use_left else IDX["right_hip"]

        shoulder = kp(xyz, vis, sh_idx)
        elbow    = kp(xyz, vis, el_idx)
        hip      = kp(xyz, vis, hi_idx)

        if any(v is None for v in [shoulder, elbow, hip]):
            ratios.append(None)
            z_offsets.append(None)
            continue

        torso_len  = dist3d(shoulder, hip)
        elbow_dist = dist3d(elbow, hip)
        ratio      = round(float(elbow_dist / torso_len), 4)
        ratios.append(ratio)

        # NEW: z-axis elbow drift (depth direction)
        # Positive = elbow forward of hip, Negative = elbow behind
        z_off = round(float((elbow[2] - hip[2]) / torso_len), 4)
        z_offsets.append(z_off)

    return ratios, z_offsets


print("compute_elbow_body_ratio_3d loaded.")

compute_elbow_body_ratio_3d loaded.


In [7]:
def compute_trunk_lean_angle(shoulder, hip):
    """
    Angle between spine vector (hip→shoulder) and vertical (Y-axis).
    Camera-angle independent — works from any view.
    
    0°   = perfectly upright
    +ve  = leaning in any direction
    threshold: flag if angle > X degrees
    """
    spine_vec = np.array(shoulder, dtype=float) - np.array(hip, dtype=float)
    vertical  = np.array([0, -1, 0])  # Y-up in MediaPipe world coords
    
    cos_a = np.dot(spine_vec, vertical) / (
        np.linalg.norm(spine_vec) * np.linalg.norm(vertical) + 1e-6
    )
    angle = float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))
    return round(angle, 2)

W0000 00:00:1775153664.668243     138 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775153664.671304     144 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775153664.816075     138 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775153664.819712     144 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [9]:
def compute_trunk_lean_3d(video_path, mode="trunk_lean"):
    deviations = []
    for frame in iter_video_frames(video_path):
        xyz, vis = extract_mp_keypoints(frame, pose_video)
        if xyz is None:
            deviations.append(None)
            continue

        use_left = pick_side(vis, IDX["left_elbow"], IDX["right_elbow"])
        sh_idx   = IDX["left_shoulder"] if use_left else IDX["right_shoulder"]
        hi_idx   = IDX["left_hip"]      if use_left else IDX["right_hip"]
        ank_idx  = IDX["left_ankle"]    if use_left else IDX["right_ankle"]

        if mode == "trunk_lean":
            shoulder = kp(xyz, vis, sh_idx)
            hip      = kp(xyz, vis, hi_idx)
            if shoulder is None or hip is None:
                deviations.append(None)
                continue

            # Camera-angle independent: angle between spine and vertical
            spine_vec = np.array(shoulder) - np.array(hip)
            vertical  = np.array([0, -1, 0])
            cos_a     = np.dot(spine_vec, vertical) / (
                np.linalg.norm(spine_vec) * np.linalg.norm(vertical) + 1e-6
            )
            angle = float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))
            sign  = 1.0 if spine_vec[0] >= 0 else -1.0
            deviations.append(round(angle * sign, 2))

        elif mode == "hip_alignment":
            shoulder = kp(xyz, vis, sh_idx)
            hip      = kp(xyz, vis, hi_idx)
            ankle    = kp(xyz, vis, ank_idx)
            if any(v is None for v in [shoulder, hip, ankle]):
                deviations.append(None)
                continue

            dev = point_to_line_dist2d(hip, shoulder, ankle)
            deviations.append(dev)

    return deviations

print("compute_trunk_lean_3d loaded.")

compute_trunk_lean_3d loaded.


In [21]:
# ── Cell 8: Run curl/pushdown reference generation ────────────
import kagglehub

hammer_videos_train , h_test  = get_videos("hammer curl")
barbell_videos_train , barbell_test = get_videos("barbell biceps curl")
tricep_videos_train , tri_test  = get_videos("tricep Pushdown")

all_videos_curl = [
    ("hammer_curl",        hammer_videos_train),
    ("barbell_biceps_curl", barbell_videos_train),
    ("tricep_pushdown",    tricep_videos_train),
]

all_ratios    = []
all_leans     = []

for exercise_name, video_list in all_videos_curl:
    print(f"\nProcessing {exercise_name}...")
    for p in video_list:
        ratios = compute_elbow_body_ratio_3d(p)
        leans          = compute_trunk_lean_3d(p, "trunk_lean")

        valid_r  = [r for r in ratios  if r is not None]
        valid_l  = [l for l in leans   if l is not None]

        print(f"  {os.path.basename(p)}: {len(valid_r)} ratio frames, "
              f"{len(valid_l)} lean frames")

        all_ratios.extend(valid_r)
        all_leans.extend(valid_l)

print(f"\nTotal valid: ratios={len(all_ratios)}, leans={len(all_leans)}")
# ── Compute thresholds ────────────────────────────────────────
fixed_ratios = []
for item in all_ratios:
    if isinstance(item, (list, tuple, np.ndarray)):
        fixed_ratios.extend([float(r) for r in item if r is not None and np.isscalar(r)])
    elif item is not None and np.isscalar(item):
        fixed_ratios.append(float(item))

all_ratios = fixed_ratios
arr_r = np.array(all_ratios)

arr_l = np.array(all_leans)

curl_pushdown_reference = {
    # Existing thresholds (same logic, 3D distances)
    "curl_pushdown":          round(float(np.percentile(arr_r, 90)), 4),
    "trunk_lean_forward_max": round(float(np.percentile(arr_l, 75)), 4), 
}

with open("curl_pushdown_reference.json", "w") as f:
    json.dump(curl_pushdown_reference, f, indent=2)

print("\ncurl_pushdown_reference.json:")
print(json.dumps(curl_pushdown_reference, indent=2))

# Distribution summaries
for name, arr in [("elbow_ratio", arr_r), ("trunk_lean_3d", arr_l),
                  ("elbow_z_drift", arr_z)]:
    print(f"\n{name}:")
    for pct, label in [(0,'min'),(25,'25th'),(50,'50th'),
                       (75,'75th'),(90,'90th'),(95,'95th'),(100,'max')]:
        fn = np.min if pct==0 else (np.max if pct==100
             else lambda a,p=pct: np.percentile(a,p))
        print(f"  {label}: {round(float(fn(arr)), 4)}")


curl_pushdown_reference.json:
{
  "curl_pushdown": 0.742,
  "trunk_lean_forward_max": 20.73
}


NameError: name 'arr_z' is not defined

---
## Part 2: Plank / Pushup — Video Dataset

### Metrics
1. **Hip alignment** (3D): 3D point-to-line distance of hip from shoulder→ankle line
2. **Shoulder angle** (3D): angle at elbow / wrist for pushup depth


In [21]:
# ── Cell 9: Plank/Pushup reference ────────────────────────────

pushup_videos_train, push_test = get_videos("push-up")
plank_videos_train , plank_test  = get_videos("plank")

all_videos_plank_pushup = [
    ("push_up", pushup_videos_train),
    ("plank",   plank_videos_train),
]

all_deviations = []

for exercise_name, video_list in all_videos_plank_pushup:
    print(f"\nProcessing {exercise_name}...")
    for p in video_list:
        devs  = compute_trunk_lean_3d(p, "hip_alignment")
        valid = [d for d in devs if d is not None]
        print(f"  {os.path.basename(p)}: {len(valid)} valid frames")
        all_deviations.extend(valid)

print(f"\nTotal valid frames: {len(all_deviations)}")

arr = np.array(all_deviations)
for pct, label in [(0,'min'),(25,'25th'),(50,'50th'),
                   (75,'75th'),(90,'90th'),(95,'95th'),(100,'max')]:
    fn = np.min if pct==0 else (np.max if pct==100
         else lambda a,p=pct: np.percentile(a,p))
    print(f"  {label}: {round(float(fn(arr)), 4)}")

plank_pushup_reference = {
    "hip_sag_max":  round(float(np.percentile(arr, 75)), 4),
    "hip_pike_min": round(float(np.percentile(arr, 25)), 4),
}

with open("plank_pushup_reference.json", "w") as f:
    json.dump(plank_pushup_reference, f, indent=2)

print("\nplank_pushup_reference.json:")
print(json.dumps(plank_pushup_reference, indent=2))


Processing push_up...
  push-up_1.mp4: 147 valid frames
  push-up_10.mp4: 10 valid frames
  push-up_11.mp4: 0 valid frames
  push-up_12.mp4: 0 valid frames
  push-up_13.mp4: 1 valid frames

Processing plank...
  plank_2.mp4: 224 valid frames
  plank_3.mp4: 71 valid frames
  plank_4.mp4: 117 valid frames
  plank_5.mp4: 752 valid frames
  plank_6.mp4: 318 valid frames

Total valid frames: 1640
  min: -0.2551
  25th: -0.0734
  50th: -0.0437
  75th: -0.0314
  90th: -0.0204
  95th: 0.0004
  max: 0.0726

plank_pushup_reference.json:
{
  "hip_sag_max": -0.0314,
  "hip_pike_min": -0.0734
}


---
## Part 3: Press — Video Dataset
*(bench press, incline bench press, shoulder press)*

### Metrics
1. **Elbow angle** (3D): angle at elbow (shoulder→elbow→wrist) at bottom of press
2. **Elbow horizontal offset** (3D): elbow position relative to shoulder in depth
   - YOLO: measured in 2D X (pixel left/right)
   - MediaPipe 3D: measured in **Z-axis depth** → "is elbow behind torso?"


In [6]:
def compute_press_metrics(video_path):
    """
    Auto-detects press type from torso angle per frame.
    
    Shoulder press (torso_angle < 45°):
      1. elbow_posterior: Z-axis offset (elbow_z - shoulder_z)
         negative = elbow behind shoulder = bad
         threshold: flag if < -0.05m (elbow drifting behind ear)
      2. elbow_flare_ratio: elbow X-spread / shoulder X-spread
         uses BOTH sides for camera-angle independence
         threshold: flag if > 1.5 (elbows too wide)
         Only measured when elbow is at or below shoulder height.

    Bench/incline (torso_angle >= 45°):
      - elbow angle at wrist (shoulder→elbow→wrist)
      - Only use frames where torso_angle >= 75° (truly horizontal)
        to filter out mixed-angle contamination from varied camera views
    """
    shoulder_flares    = []
    shoulder_posterior = []
    bench_angles       = []

    for frame in iter_video_frames(video_path):
        xyz, vis = extract_mp_keypoints(frame, pose_video)
        if xyz is None:
            shoulder_flares.append(None)
            shoulder_posterior.append(None)
            bench_angles.append(None)
            continue

        use_left = pick_side(vis, IDX["left_elbow"], IDX["right_elbow"])
        sh_idx   = IDX["left_shoulder"] if use_left else IDX["right_shoulder"]
        el_idx   = IDX["left_elbow"]    if use_left else IDX["right_elbow"]
        hi_idx   = IDX["left_hip"]      if use_left else IDX["right_hip"]
        wr_idx   = IDX["left_wrist"]    if use_left else IDX["right_wrist"]

        shoulder = kp(xyz, vis, sh_idx)
        elbow    = kp(xyz, vis, el_idx)
        hip      = kp(xyz, vis, hi_idx)
        wrist    = kp(xyz, vis, wr_idx)

        if any(v is None for v in [shoulder, elbow, hip]):
            shoulder_flares.append(None)
            shoulder_posterior.append(None)
            bench_angles.append(None)
            continue

        # ── Detect press type ─────────────────────────────────
        spine_vec   = np.array(shoulder) - np.array(hip)
        vertical    = np.array([0, -1, 0])
        cos_a       = np.dot(spine_vec, vertical) / (
            np.linalg.norm(spine_vec) * np.linalg.norm(vertical) + 1e-6
        )
        torso_angle = float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))

        if torso_angle < 45:
            # ── Shoulder press ────────────────────────────────
            bench_angles.append(None)

            # 1. Posterior elbow drift (Z-axis, camera independent)
            post = round(float(elbow[2] - shoulder[2]), 4)
            shoulder_posterior.append(post)

            # 2. Elbow flare ratio — only when elbow at/below shoulder
            # Uses both sides for camera independence
            if elbow[1] > shoulder[1] - 0.03:
                l_el = kp(xyz, vis, IDX["left_elbow"])
                r_el = kp(xyz, vis, IDX["right_elbow"])
                l_sh = kp(xyz, vis, IDX["left_shoulder"])
                r_sh = kp(xyz, vis, IDX["right_shoulder"])
                if all(v is not None for v in [l_el, r_el, l_sh, r_sh]):
                    elbow_w    = abs(float(l_el[0] - r_el[0]))
                    shoulder_w = abs(float(l_sh[0] - r_sh[0]))
                    flare_ratio = round(elbow_w / (shoulder_w + 1e-6), 4)
                    shoulder_flares.append(flare_ratio)
                else:
                    shoulder_flares.append(None)
            else:
                shoulder_flares.append(None)

        else:
            # ── Bench / incline press ─────────────────────────
            shoulder_flares.append(None)
            shoulder_posterior.append(None)

            # Only use truly horizontal frames (>= 75°) to avoid
            # mixed camera angle contamination in the dataset
            if torso_angle >= 75 and wrist is not None:
                bench_angles.append(round(angle3d(shoulder, elbow, wrist), 2))
            else:
                bench_angles.append(None)

    return shoulder_flares, shoulder_posterior, bench_angles

In [7]:
# ── Cell 11: Run press reference generation ───────────────────
bench_videos    = get_videos("bench press")
incline_videos  = get_videos("incline bench press")
shoulder_videos = get_videos("shoulder press")

all_press_videos = [
    ("bench_press",         bench_videos),
    ("incline_bench_press", incline_videos),
    ("shoulder_press",      shoulder_videos),
]

all_flares       = []
all_posterior    = []
all_bench_angles = []

for exercise_name, video_list in all_press_videos:
    print(f"\nProcessing {exercise_name}...")
    for p in video_list[0]:
        flares, posterior, bench = compute_press_metrics(p)
        vf = [x for x in flares    if x is not None]
        vp = [x for x in posterior if x is not None]
        vb = [x for x in bench     if x is not None]
        print(f"  {os.path.basename(p)}: {len(vf)} flare frames, "
              f"{len(vp)} posterior frames, {len(vb)} bench frames")
        all_flares.extend(vf)
        all_posterior.extend(vp)
        all_bench_angles.extend(vb)

print("\n=== SHOULDER PRESS — elbow flare ratio (elbow_width / shoulder_width) ===")
if all_flares:
    arr_f = np.array(all_flares)
    for pct, label in [(0,"min"),(25,"25th"),(50,"50th"),(75,"75th"),(90,"90th"),(95,"95th"),(100,"max")]:
        fn = np.min if pct==0 else (np.max if pct==100 else lambda a,p=pct: np.percentile(a,p))
        print(f"  {label}: {round(float(fn(arr_f)), 4)}")

print("\n=== SHOULDER PRESS — elbow posterior drift (elbow_z - shoulder_z) ===")
if all_posterior:
    arr_p = np.array(all_posterior)
    for pct, label in [(0,"min"),(25,"25th"),(50,"50th"),(75,"75th"),(90,"90th"),(95,"95th"),(100,"max")]:
        fn = np.min if pct==0 else (np.max if pct==100 else lambda a,p=pct: np.percentile(a,p))
        print(f"  {label}: {round(float(fn(arr_p)), 4)}")

print("\n=== BENCH / INCLINE — elbow angle at wrist (horizontal frames only) ===")
if all_bench_angles:
    arr_b = np.array(all_bench_angles)
    for pct, label in [(0,"min"),(25,"25th"),(50,"50th"),(75,"75th"),(90,"90th"),(95,"95th"),(100,"max")]:
        fn = np.min if pct==0 else (np.max if pct==100 else lambda a,p=pct: np.percentile(a,p))
        print(f"  {label}: {round(float(fn(arr_b)), 2)}")

press_reference = {}
if all_flares:
    # 90th pct = only flag truly excessive flare
    press_reference["shoulder_flare_max"] = round(float(np.percentile(arr_f, 90)), 4)
if all_posterior:
    # 10th pct = elbow drift threshold (negative = behind shoulder)
    press_reference["elbow_behind_threshold"] = round(float(np.percentile(arr_p, 10)), 4)
if all_bench_angles:
    # 10th pct = minimum safe elbow angle
    press_reference["bench_elbow_angle_min"] = round(float(np.percentile(arr_b, 10)), 2)

with open("press_reference.json", "w") as f:
    json.dump(press_reference, f, indent=2)
print("\npress_reference.json:")
print(json.dumps(press_reference, indent=2))


Processing bench_press...


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


  bench press_1.mp4: 0 flare frames, 0 posterior frames, 0 bench frames
  bench press_10.mp4: 0 flare frames, 0 posterior frames, 42 bench frames
  bench press_11.mp4: 0 flare frames, 0 posterior frames, 39 bench frames
  bench press_12.mp4: 0 flare frames, 0 posterior frames, 37 bench frames
  bench press_13.mp4: 0 flare frames, 0 posterior frames, 74 bench frames

Processing incline_bench_press...
  incline bench press_1.mp4: 0 flare frames, 0 posterior frames, 1 bench frames
  incline bench press_10.mp4: 0 flare frames, 0 posterior frames, 9 bench frames
  incline bench press_11.mp4: 0 flare frames, 0 posterior frames, 20 bench frames
  incline bench press_12.mp4: 0 flare frames, 2 posterior frames, 5 bench frames
  incline bench press_13.mp4: 0 flare frames, 1 posterior frames, 0 bench frames

Processing shoulder_press...
  shoulder press_10.mp4: 91 flare frames, 112 posterior frames, 0 bench frames
  shoulder press_11.mp4: 20 flare frames, 52 posterior frames, 0 bench frames
  sho

---
## Part 4: Squat & Deadlift — Frames Dataset
*(squat, front_squat, zercher_squat, deadlift, romanian_deadlift, sumo_deadlift)*

**Uses the Kaggle frames dataset** (`pratapdevs11/gym-data-coco-format-for-yolo-pose`),
not the video dataset. Same as YOLO notebook Cell 23-29.

### Metrics (all upgraded to 3D)
1. **Hip angle** (3D): shoulder→hip→knee angle
2. **Knee angle** (3D): hip→knee→ankle angle
3. **Asymmetry**: |knee_angle_L - knee_angle_R|
4. **Knee-ankle ratio** (3D): `dist3d(L_knee,R_knee) / dist3d(L_ankle,R_ankle)`
5. **Trunk lean** (3D depth Z)
6. **Phase detection** (TOP/MID/BOTTOM by knee angle percentile)
7. **Knee collapse ratio** (bottom/top phase knee-ankle ratio)
8. **NEW — Forward lean Z**: `(shoulder_z - hip_z)` at each phase


In [13]:
# ── Cell 12: Download frames dataset ──────────────────────────
import kagglehub

frames_dataset_path = kagglehub.dataset_download(
    "pratapdevs11/gym-data-coco-format-for-yolo-pose"
)
print("Path to dataset files:", frames_dataset_path)

DATASET_ROOT = os.path.join(frames_dataset_path, 'JIM_DATA29')
OUTPUT_DIR   = "/kaggle/working/references"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXERCISES = {
    "deadlift":          os.path.join(DATASET_ROOT, "Deadlifts/deadlift Data29"),
    "romanian_deadlift": os.path.join(DATASET_ROOT, "Deadlifts/romania Data29"),
    "sumo_deadlift":     os.path.join(DATASET_ROOT, "Deadlifts/sumo data 29"),
    "squat":             os.path.join(DATASET_ROOT, "Squats/squats data29"),
    "front_squat":       os.path.join(DATASET_ROOT, "Squats/front_squat data29"),
    "zercher_squat":     os.path.join(DATASET_ROOT, "Squats/zercher Data29"),
}

print("\nExercise folders:")
for name, p in EXERCISES.items():
    if os.path.exists(p):
        if( name == "deadlift"):
            train_path=os.path.join(p,"image","Train")
        else:
            train_path=os.path.join(p,"images","train")
        count = sum(1 for f in os.listdir(train_path))
        print(f"  {name}: {count} train images")
      
    else:
        print(f"  {name}: NOT FOUND")

Path to dataset files: /kaggle/input/datasets/pratapdevs11/gym-data-coco-format-for-yolo-pose

Exercise folders:
  deadlift: 516 train images
  romanian_deadlift: 315 train images
  sumo_deadlift: 510 train images
  squat: 492 train images
  front_squat: 630 train images
  zercher_squat: 411 train images


In [15]:
# ── Cell 13: Keypoint extraction for frames dataset ───────────

def extract_keypoints_from_frames(exercise_name, exercise_path):
    """
    3D UPGRADE of extract_keypoints (YOLO notebook Cell 26).
    Runs MediaPipe on all train images.
    Returns list of dicts with xyz (world 3D) and vis per keypoint.
    """
    frames_data = []
    images = list(iter_images(exercise_path))
    print(f"\n{exercise_name}: {len(images)} images")

    for i, img_path in enumerate(images):
        if i % 100 == 0:
            print(f"  {i}/{len(images)}...")
        try:
            img_bgr = cv2.imread(img_path)
            if img_bgr is None:
                continue

            xyz, vis = extract_mp_keypoints(img_bgr, pose_static)
            if xyz is None:
                continue

            frames_data.append({
                "xyz":      xyz,      # [33,3] world coords
                "vis":      vis,      # [33] visibility
                "path":     img_path
            })
        except Exception as e:
            print(f"  ERROR {img_path}: {e}")

    print(f"  → {len(frames_data)} valid frames")
    return frames_data


print("extract_keypoints_from_frames loaded.")

extract_keypoints_from_frames loaded.


In [10]:
# ── Cell 14: 3D metrics computation for squat/deadlift ─────────

def compute_metrics_3d(frames_data):
    """
    3D UPGRADE of compute_metrics (YOLO notebook Cell 27).
    
    All distances: dist3d (sqrt(dx²+dy²+dz²))
    All angles: angle3d (3D dot product)
    Trunk lean: Z-depth based instead of X-pixel based
    
    NEW metrics added:
      - trunk_lean_z_at_phase: lean per phase (TOP/MID/BOTTOM)
      - hip_depth_lean: forward lean using Z-axis
    """
    all_values = defaultdict(list)
    temp_knee_angles = []
    temp_knee_ratios = []
    temp_trunk_leans = []

    for frame in frames_data:
        xyz = frame["xyz"]
        vis = frame["vis"]

        def kp_f(idx, thr=CONF_THRESHOLD):
            return kp(xyz, vis, idx, thr)

        shL  = kp_f(IDX["left_shoulder"])
        shR  = kp_f(IDX["right_shoulder"])
        hipL = kp_f(IDX["left_hip"])
        hipR = kp_f(IDX["right_hip"])
        kneL = kp_f(IDX["left_knee"])
        kneR = kp_f(IDX["right_knee"])
        ankL = kp_f(IDX["left_ankle"])
        ankR = kp_f(IDX["right_ankle"])

        # ── Hip angles (3D) ───────────────────────────────────
        if all(v is not None for v in [shL, hipL, kneL]):
            all_values["hip_angle_L"].append(angle3d(shL, hipL, kneL))
        if all(v is not None for v in [shR, hipR, kneR]):
            all_values["hip_angle_R"].append(angle3d(shR, hipR, kneR))

        # ── Knee angles (3D) ──────────────────────────────────
        kaL, kaR = None, None
        if all(v is not None for v in [hipL, kneL, ankL]):
            kaL = angle3d(hipL, kneL, ankL)
            all_values["knee_angle_L"].append(kaL)
        if all(v is not None for v in [hipR, kneR, ankR]):
            kaR = angle3d(hipR, kneR, ankR)
            all_values["knee_angle_R"].append(kaR)

        knee_angle_avg = None
        if kaL is not None and kaR is not None:
            knee_angle_avg = (kaL + kaR) / 2.0
        elif kaL is not None:
            knee_angle_avg = kaL
        elif kaR is not None:
            knee_angle_avg = kaR

        # ── Asymmetry ─────────────────────────────────────────
        if kaL is not None and kaR is not None:
            all_values["asymmetry"].append(abs(kaL - kaR))

        # ── Knee-ankle ratio (3D distances) ───────────────────
        kneL_h = kp_f(IDX["left_knee"],   0.5)
        kneR_h = kp_f(IDX["right_knee"],  0.5)
        ankL_h = kp_f(IDX["left_ankle"],  0.5)
        ankR_h = kp_f(IDX["right_ankle"], 0.5)

        knee_ratio = None
        if all(v is not None for v in [kneL_h, kneR_h, ankL_h, ankR_h]):
            ankle_w    = dist3d(ankL_h, ankR_h)
            knee_w     = dist3d(kneL_h, kneR_h)
            knee_ratio = round(float(knee_w / ankle_w), 4)
            all_values["knee_ankle_ratio"].append(knee_ratio)

        # ── Trunk lean (3D Z-axis) ────────────────────────────
        # UPGRADE: was X-pixel lean, now Z-depth lean
        use_left = vis[IDX["left_hip"]] >= vis[IDX["right_hip"]]
        sh_h  = kp_f(IDX["left_shoulder"] if use_left else IDX["right_shoulder"], 0.5)
        hip_h = kp_f(IDX["left_hip"]      if use_left else IDX["right_hip"],      0.5)

        trunk_lean = None
        if sh_h is not None and hip_h is not None:
            torso_len  = dist3d(sh_h, hip_h)
            # Positive = shoulder forward (in Z) = leaning forward
            trunk_lean = round(float((hip_h[2] - sh_h[2]) / torso_len), 4)
            all_values["trunk_lean"].append(trunk_lean)

        # Store for phase analysis
        if knee_angle_avg is not None:
            temp_knee_angles.append(knee_angle_avg)
            temp_knee_ratios.append(knee_ratio)
            temp_trunk_leans.append(trunk_lean)

        # ── Back rounding (Z-axis spinal curvature) ───────────
        back_round = None
        if all(v is not None for v in [shL, shR, hipL, hipR, kneL, kneR]):
            sh_mid  = (np.array(shL)  + np.array(shR))  / 2.0
            hip_mid = (np.array(hipL) + np.array(hipR)) / 2.0
            kne_mid = (np.array(kneL) + np.array(kneR)) / 2.0
            upper_spine = hip_mid - sh_mid
            lower_spine = kne_mid - hip_mid
            torso_len_b = np.linalg.norm(sh_mid - hip_mid) + 1e-6
            back_round  = round(float((upper_spine[2] - lower_spine[2]) / torso_len_b), 4)
            all_values["back_round"].append(back_round)

    # ── Phase detection (same logic as YOLO notebook) ─────────
    if len(temp_knee_angles) > 0:
        angles = np.array(temp_knee_angles)
        low_thr  = np.percentile(angles, 20)
        high_thr = np.percentile(angles, 80)

        top_ratios = []; mid_ratios = []; bottom_ratios = []
        top_leans  = []; mid_leans  = []; bottom_leans  = []

        for angle, ratio, lean in zip(temp_knee_angles,
                                      temp_knee_ratios,
                                      temp_trunk_leans):
            if ratio is None:
                continue
            if angle <= low_thr:
                bottom_ratios.append(ratio)
                if lean is not None: bottom_leans.append(lean)
            elif angle >= high_thr:
                top_ratios.append(ratio)
                if lean is not None: top_leans.append(lean)
            else:
                mid_ratios.append(ratio)
                if lean is not None: mid_leans.append(lean)

        if top_ratios:    all_values["knee_ratio_top"]    = top_ratios
        if mid_ratios:    all_values["knee_ratio_mid"]    = mid_ratios
        if bottom_ratios: all_values["knee_ratio_bottom"] = bottom_ratios

        # NEW: trunk lean per phase (helps set phase-specific lean thresholds)
        if top_leans:    all_values["trunk_lean_top"]    = top_leans
        if mid_leans:    all_values["trunk_lean_mid"]    = mid_leans
        if bottom_leans: all_values["trunk_lean_bottom"] = bottom_leans

        # Knee collapse ratio
        if top_ratios and bottom_ratios:
            collapse = float(np.mean(bottom_ratios) / (np.mean(top_ratios) + 1e-6))
            all_values["knee_collapse_ratio"].append(round(collapse, 4))

    return all_values


print("compute_metrics_3d loaded.")

compute_metrics_3d loaded.


In [21]:
# ── Cell 15: Reference generator for squat/deadlift ───────────

def summarize(values):
    arr = np.array(values)
    return {
        "count": int(arr.size),
        "min":   round(float(np.min(arr)), 4),
        "25th":  round(float(np.percentile(arr, 25)), 4),
        "50th":  round(float(np.percentile(arr, 50)), 4),
        "75th":  round(float(np.percentile(arr, 75)), 4),
        "90th":  round(float(np.percentile(arr, 90)), 4),
        "95th":  round(float(np.percentile(arr, 95)), 4),
        "max":   round(float(np.max(arr)), 4),
    }


def generate_reference_3d(all_values, exercise_name):
    """
    3D UPGRADE of generate_reference (YOLO notebook Cell 28).
    Same percentile strategy, now includes:
      - Z-axis trunk lean thresholds
      - Phase-specific lean (new)
      - All existing keys preserved for backward compat
    """
    ref = {"exercise": exercise_name}

    # Hip angle
    hip_vals = []
    for side in ["hip_angle_L", "hip_angle_R"]:
        if all_values.get(side): hip_vals.extend(all_values[side])
    if hip_vals:
        arr = np.array(hip_vals)
        ref["hip_angle_min"]     = round(float(np.percentile(arr, 25)), 2)
        ref["hip_angle_summary"] = summarize(hip_vals)

    # Knee angle
    knee_vals = []
    for side in ["knee_angle_L", "knee_angle_R"]:
        if all_values.get(side): knee_vals.extend(all_values[side])
    if knee_vals:
        arr = np.array(knee_vals)
        ref["knee_angle_min"]     = round(float(np.percentile(arr, 25)), 2)
        ref["knee_angle_summary"] = summarize(knee_vals)

    # Asymmetry
    if all_values.get("asymmetry"):
        arr = np.array(all_values["asymmetry"])
        ref["asymmetry_max"]     = round(float(np.percentile(arr, 90)), 2)
        ref["asymmetry_summary"] = summarize(all_values["asymmetry"])

    # Knee-ankle ratio
    if all_values.get("knee_ankle_ratio"):
        arr = np.array(all_values["knee_ankle_ratio"])
        ref["knee_ankle_ratio_min"]     = round(float(np.percentile(arr, 25)), 4)
        ref["knee_ankle_ratio_summary"] = summarize(all_values["knee_ankle_ratio"])

    # Trunk lean (3D Z-based)
    if all_values.get("trunk_lean"):
        arr = np.array(all_values["trunk_lean"])
        ref["trunk_lean_max"]     = round(float(np.percentile(arr, 90)), 4)
        ref["trunk_lean_min"]     = round(float(np.percentile(arr, 10)), 4)
        ref["trunk_lean_summary"] = summarize(all_values["trunk_lean"])
    if "back_round" in all_values:
        arr_back = np.array(all_values["back_round"])
        ref["back_round_max"] = round(float(np.percentile(arr_back, 90)), 4)
        print(f"\nback_round:")
        for pct, label in [(0,'min'),(25,'25th'),(50,'50th'),(75,'75th'),(90,'90th'),(95,'95th'),(100,'max')]:
            fn = np.min if pct==0 else (np.max if pct==100 else lambda a,p=pct: np.percentile(a,p))
            print(f"  {label}: {round(float(fn(arr_back)), 4)}")
    # Phase-specific trunk lean (NEW)
    for phase in ["top", "mid", "bottom"]:
        key = f"trunk_lean_{phase}"
        if all_values.get(key):
            ref[f"trunk_lean_{phase}_summary"] = summarize(all_values[key])

    # Knee collapse ratio
    if all_values.get("knee_collapse_ratio"):
        arr = np.array(all_values["knee_collapse_ratio"])
        ref["knee_collapse_ratio_min"]     = round(float(np.percentile(arr, 25)), 4)
        ref["knee_collapse_ratio_summary"] = summarize(all_values["knee_collapse_ratio"])

    # Phase knee ratios
    for phase in ["top", "mid", "bottom"]:
        key = f"knee_ratio_{phase}"
        if all_values.get(key):
            ref[f"knee_ratio_{phase}_summary"] = summarize(all_values[key])

    return ref


print("generate_reference_3d loaded.")

generate_reference_3d loaded.


In [22]:
# ── Cell 16: Run full squat/deadlift pipeline ─────────────────

all_references = {}

for exercise_name, exercise_path in EXERCISES.items():
    if not os.path.exists(exercise_path):
        print(f"\nSkipping {exercise_name} — path not found")
        continue

    # Step 1: Extract keypoints from images
    frames_data = extract_keypoints_from_frames(exercise_name, exercise_path)

    if not frames_data:
        print(f"  No valid frames for {exercise_name}")
        continue

    # Step 2: Compute 3D metrics
    all_values = compute_metrics_3d(frames_data)

    # Step 3: Generate reference
    ref = generate_reference_3d(all_values, exercise_name)
    all_references[exercise_name] = ref

    # Step 4: Save
    out_path = os.path.join(OUTPUT_DIR, f"{exercise_name}_reference.json")
    with open(out_path, "w") as f:
        json.dump(ref, f, indent=2)

    print(f"\n{'='*50}")
    print(f"Reference: {exercise_name}")
    print(f"{'='*50}")
    for key, val in ref.items():
        if not key.endswith("_summary") and key != "exercise":
            print(f"  {key}: {val}")

print("\nAll references saved to:", OUTPUT_DIR)


deadlift: 1636 images
  0/1636...
  100/1636...
  200/1636...
  300/1636...
  400/1636...
  500/1636...
  600/1636...
  700/1636...
  800/1636...
  900/1636...
  1000/1636...
  1100/1636...
  1200/1636...
  1300/1636...
  1400/1636...
  1500/1636...
  1600/1636...
  → 818 valid frames

back_round:
  min: 0.2494
  25th: 0.3762
  50th: 0.8728
  75th: 1.6132
  90th: 1.6751
  95th: 1.6965
  max: 1.763

Reference: deadlift
  hip_angle_min: 68.55
  knee_angle_min: 97.74
  asymmetry_max: 21.81
  knee_ankle_ratio_min: 0.7968
  trunk_lean_max: 0.9468
  trunk_lean_min: 0.3417
  back_round_max: 1.6751
  knee_collapse_ratio_min: 1.3951

romanian_deadlift: 315 images
  0/315...
  100/315...
  200/315...
  300/315...
  → 315 valid frames

back_round:
  min: 0.1525
  25th: 0.2501
  50th: 0.5974
  75th: 0.8964
  90th: 1.176
  95th: 1.331
  max: 1.4704

Reference: romanian_deadlift
  hip_angle_min: 118.64
  knee_angle_min: 144.3
  asymmetry_max: 16.23
  knee_ankle_ratio_min: 0.7268
  trunk_lean_max: 0

In [ ]:
# ── Cell 17: Print full distributions ─────────────────────────
# Same as YOLO notebook Cell 29, extended with 3D phase lean

for exercise_name, ref in all_references.items():
    print(f"\n{'='*50}")
    print(f"Full distributions: {exercise_name}")
    print(f"{'='*50}")

    if "knee_collapse_ratio_summary" in ref:
        print("\n🔥 Knee Collapse (MOST IMPORTANT):")
        for k, v in ref["knee_collapse_ratio_summary"].items():
            print(f"    {k}: {v}")

    for phase in ["top", "mid", "bottom"]:
        key = f"knee_ratio_{phase}_summary"
        if key in ref:
            print(f"\n📊 Knee Ratio ({phase.upper()} phase):")
            for k, v in ref[key].items():
                print(f"    {k}: {v}")

    # NEW: trunk lean per phase
    for phase in ["top", "mid", "bottom"]:
        key = f"trunk_lean_{phase}_summary"
        if key in ref:
            print(f"\n📐 Trunk Lean 3D ({phase.upper()} phase):")
            for k, v in ref[key].items():
                print(f"    {k}: {v}")

    for key, val in ref.items():
        if key.endswith("_summary") and not any(x in key for x in [
            "knee_collapse_ratio", "knee_ratio_top", "knee_ratio_mid",
            "knee_ratio_bottom", "trunk_lean_top", "trunk_lean_mid", "trunk_lean_bottom"
        ]):
            metric_name = key.replace("_summary", "")
            print(f"\n  {metric_name}:")
            for k, v in val.items():
                print(f"    {k}: {v}")

In [ ]:
# ── Cell 18: Cleanup ──────────────────────────────────────────
pose_static.close()
pose_video.close()
print("MediaPipe instances closed.")
print("\nGenerated reference files:")
for f in os.listdir(OUTPUT_DIR):
    print(f"  {f}")
# Also list working dir
for f in ["curl_pushdown_reference.json",
          "plank_pushup_reference.json",
          "press_reference.json"]:
    if os.path.exists(f):
        print(f"  {f}")